In [ ]:

for index, row in filtered_df.iterrows():
    fasta_filename = row['filename']
    print(f"Processing file: {fasta_filename}")
    start_index = row['start_position_on_the_genomic_accession']
    end_index = row['end_position_on_the_genomic_accession']
    uniprot = row['Uniprot']
    entry = row['Entry']
    drug = row['drug_full']
    gene_name = row['gene_name']
    orientation = row['orientation']
    file_path = os.path.join(fasta_dir, fasta_filename)
    filename = os.path.basename(file_path)
    alignment = load_alignment(file_path)
    print(f"Loaded alignment matrix for {filename}")
    print(f"Gene: {gene_name}, Drug: {drug}")
    print(f"Gene shape: {alignment.matrix.shape}")
    h37rv_numbers = np.load("/work/pi_annagreen_umass_edu/mahbuba/MTB-CNN/output_data/X_matrix_H37RV_coords.npy")
    h37rv_ref = alignment.select(sequences=[alignment.id_to_index["MT_H37Rv"]])
    subset_alignment, column_index, start_index, end_index = sort_gene_indices(h37rv_numbers, start_index, end_index, alignment)
    h37rv_alignment = select_subset_alignment(h37rv_ref, start_index, end_index, h37rv_numbers[:, column_index])
    h37rv_nongap_indices = np.where(h37rv_alignment[0] != "-")[0]
    h37rv_sequence_str = ''.join(h37rv_alignment[0][h37rv_nongap_indices])

    results = []  # Initialize as a list
    X, y, h37rv_protein, wildtype_labels, identifiers = create_feature_matrix(subset_alignment=subset_alignment,drug=drug,phenotype_data=phenotype_data,orientation=orientation,h37rv_nongap_indices=h37rv_nongap_indices,
    h37rv_sequence_str=h37rv_sequence_str)
    # Save sequences to FASTA for ESM
    fasta_filename = f"{gene_name}_esm_input.fasta"
    with open(fasta_filename, 'w') as fasta_file:
        for identifier, sequence in zip(identifiers, X):
            fasta_file.write(f">{identifier}\n{sequence}\n")
    print("aa check after translation", X[0]==h37rv_protein)

    # Original lengths
    print(f"Original feature matrix length: {X.shape[0]}")
    print(f"Original labels length: {len(wildtype_labels)}")

    X_unique_rows, unique_indices = drop_identical_sequences(X)
    wildtype_labels_filtered = [wildtype_labels[i] for i in unique_indices]
    ys = y[unique_indices]
    identifiers_filtered = [identifiers[i] for i in unique_indices]

    # Further details
    print(f"Number of features (original): {X.shape}")
    print(f"Number of features (unique): {X_unique_rows.shape}")
    print(f"Filtered labels length: {len(wildtype_labels_filtered)}")

    # Check if the lengths match
    if len(X_unique_rows) != len(wildtype_labels_filtered):
        print("Length mismatch detected.")
    else:
        print("Lengths match.")



    phenotypes, [X_unique_rows_filtered] = filter_nan_labels(ys, X_unique_rows)
    # Filter out 'nan' labels from wildtype_labels_filtered using the same indices
    _, [wildtype_labels_filtered] = filter_nan_labels(ys, wildtype_labels_filtered)

    _, [identifiers_filtered] = filter_nan_labels(ys, identifiers_filtered)

    # Check the filtered lengths
    print(f"Filtered phenotypes length: {len(phenotypes)}")
    print(f"Filtered X_unique_rows length: {len(X_unique_rows_filtered)}")
    print(f"Filtered wildtype_labels length: {len(wildtype_labels_filtered)}")

    wildtype_positions = [index for index, label in enumerate(wildtype_labels_filtered) if label == "WT"]
    if wildtype_positions:
        print(f"'wildtype' found at positions: {wildtype_positions}")
    else:
        print("The list does not contain 'wildtype'.")


    # Collect data into a DataFrame
    data = {
        'Identifier': identifiers,
        'Sequence': X,
        'Phenotype': y,
        'Wildtype_Status': wildtype_labels
    }
    df = pd.DataFrame(data)
    csv_filename = f"{gene_name}_esm_data_table.csv"
    df.to_csv(csv_filename, index=False)

    # Data table information
    print(f"Data table information: {gene_name}")
    print("Total no of sequences:", X.shape[0])
    print("Total no of unique sequences:", X_unique_rows.shape[0])
    gene_length_nt = end_index - start_index
    print("Gene length (nt):", gene_length_nt)
    print("No of variable positions:", len(h37rv_nongap_indices))

    # Append data to the data_table as a dictionary
    gene_data = {
        'Gene': gene_name,
        'Total no of sequences': X.shape[0],
        'Total no of unique sequences':X_unique_rows.shape[0],
        'Gene length (nt)': int(gene_length_nt),
        'No of variable positions': len(h37rv_nongap_indices)
    }
    data_table.append(gene_data)

In [ ]:
np.array(X_unique_rows_filtered).shape

In [ ]:
wildtype_representations, mutated_representations , mutations_all = generate_embeddings(X_unique_rows_filtered, phenotypes, h37rv_protein, wildtype_labels_filtered)

In [ ]:

def create_dataframe(wildtype_representations, mutated_representations, phenotypes, identifiers_filtered, mutations_all,gene_name):
    # Flatten embeddings and create labels
    embeddings = []
    labels = []
    all_mutations = []

    # Add mutated embeddings and mutations
    for i, mutated_embedding in enumerate(mutated_representations):
        embeddings.append(mutated_embedding)
        labels.append("mutated")
        all_mutations.append(mutations_all[i])

    # Add wildtype data
    embeddings.append(wildtype_representations[0])
    labels.append("wildtype")
    all_mutations.append([])  # No mutations for wildtype

    delta_z_scores = [
        np.linalg.norm(mutated_embedding - wildtype_representations[0])
        for mutated_embedding in mutated_representations
    ]

    delta_z_scores.append(0)  # Add 0 for wildtype
    
    print(len(identifiers_filtered), len(embeddings), len(labels), len(delta_z_scores),len(phenotypes),len(all_mutations))


    data = {
        "identifier": identifiers_filtered,
        "embedding": embeddings,
        "type_label": labels,
        "delta_z_score": delta_z_scores,
        "phenotype": phenotypes,
        "mutation": all_mutations  # Add the mutations column
    }

    df = pd.DataFrame(data)

    return df


In [ ]:
results_df = create_dataframe(wildtype_representations, mutated_representations, phenotypes, identifiers_filtered, mutations_all,gene_name)

In [ ]:
results_df['type_label'].value_counts()

In [ ]:
results_df

In [ ]:

results_csv_path = f'/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/esm/{gene_name}_deltaz.csv'
results_df.to_csv(results_csv_path, index=False)

In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
import ast

def extract_mutation_info(mutation):
    """
    Extract wild-type residue, position, and mutant residue from mutation string like 'p.XnnY'.
    Assumes mutation is in the format 'p.XnnY' where:
    - X is the wild-type residue
    - nn is the position (adjusted to zero-based indexing)
    - Y is the mutant residue
    """
    try:
        if mutation.startswith('p.') and len(mutation) > 4:
            wt_residue = mutation[2]  # Extract wild-type residue (after 'p.')
            position = int(mutation[3:-1]) - 1  # Extract position and adjust to zero-based index
            mt_residue = mutation[-1]  # Extract mutant residue
            return wt_residue, position, mt_residue
        else:
            raise ValueError(f"Invalid mutation format: {mutation}")
    except Exception as e:
        raise ValueError(f"Error parsing mutation '{mutation}': {e}")




import torch
from transformers import AutoTokenizer, EsmForMaskedLM
import ast

def compute_llr_multiple_mutations(df, protein_sequence):
    # Load the ESM-2 model and tokenizer
    model_name = "facebook/esm2_t30_150M_UR50D"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = EsmForMaskedLM.from_pretrained(model_name).to(torch.device("cuda"))

    # Tokenize the input sequence
    input_ids = tokenizer.encode(str(protein_sequence), return_tensors="pt").to(torch.device("cuda"))
    sequence_length = input_ids.shape[1] - 2  # Exclude special tokens

    # List to store LLR scores
    llr_scores = []

    for index, row in df.iterrows():
        mutation_list = row['mutation']  # Get the mutation list (as a string)
        # print(f"Processing row {index} with mutations: {mutation_list}")
        total_llr = 0  # Initialize total LLR for this row

        # Skip empty mutation lists or empty strings
        if not mutation_list or mutation_list == []:
            print(f"No mutations for row {index}, skipping.")
            llr_scores.append(float('nan'))  # Add NaN if no mutation
            continue

        # Convert the string to a list if it is a string representation of a list
        try:
            mutation_list = ast.literal_eval(mutation_list)  # Convert string to list
        except (ValueError, SyntaxError) as e:
            print(f"Error converting mutation list for row {index}: {e}")
            llr_scores.append(float('nan'))
            continue

        # Mask all mutation positions at once
        masked_input_ids = input_ids.clone()

        # Extract mutation info for all mutations in the list
        mutation_positions = []
        for mutation in mutation_list:
            mutation = mutation.strip("[]").strip("'")  # Clean the mutation string

            # Check if the mutation starts with 'p.'
            if mutation.startswith('p.'):
                try:
                    wt_residue, position, mt_residue = extract_mutation_info(mutation)
                    mutation_positions.append((wt_residue, position, mt_residue))
                except Exception as e:
                    print(f"Error processing mutation {mutation}: {e}")
                    llr_scores.append(float('nan'))
                    continue
                    

        # Mask all mutated positions in the sequence
        for wt_residue, position, mt_residue in mutation_positions:
            # adjusted_position = position-1+1 
            # -1 for 0 index
            # +1 for cls token
            if  position>= sequence_length:
                print(f"Skipping mutation {mutation} at position {position}, out of bounds.")
                continue
            masked_input_ids[0, position] = tokenizer.mask_token_id  

        # Forward pass with masked input
        with torch.no_grad():
            logits = model(masked_input_ids).logits

        # Compute LLR for each mutation in the list
        for wt_residue, position, mt_residue in mutation_positions:
            if position >= sequence_length:
                continue

            # Calculate log probabilities
            probabilities = torch.nn.functional.softmax(logits[0, position], dim=0).to(torch.device("cuda"))  
            log_probabilities = torch.log(probabilities)

            # Get the log probability of the wild-type residue
            log_prob_wt = log_probabilities[tokenizer.convert_tokens_to_ids(wt_residue)].item()

            # Get the log probability of the mutant residue
            log_prob_mt = log_probabilities[tokenizer.convert_tokens_to_ids(mt_residue)].item()

            # Calculate LLR for the mutant residue
            llr_score = log_prob_mt - log_prob_wt
            # print(f"LLR Score for {wt_residue}{position + 1}{mt_residue}: {llr_score}")
            total_llr += llr_score

        # Append total LLR for this row
        llr_scores.append(total_llr)

    print("Length of total computations:", len(llr_scores))

    # Add LLR scores to the DataFrame
    df['llr_score'] = llr_scores

    return df


In [ ]:

deltaz_df = pd.read_csv(f'/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/esm/{gene_name}_deltaz.csv')
ref_seqs=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/protein_sequences.csv')
protein_sequence = ref_seqs.loc[ref_seqs['gene'] == gene_name, 'protein_sequence'].values

In [ ]:
protein_sequence[0]

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer
model_name = "facebook/esm2_t30_150M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize the sequence and inspect the tokens
# protein_sequence = "MKTIIALSYIFCLVFA"  # Example sequence
input_ids = tokenizer.encode(protein_sequence[0], return_tensors="pt")
print("Tokenized sequence:", input_ids)

# Convert token IDs back to tokens to verify if special tokens are added
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
print("Tokens:", tokens)


In [ ]:

results_df = compute_llr_multiple_mutations(deltaz_df, protein_sequence)


In [ ]:
results_df['gene_name'] = gene_name

In [ ]:
results_df

In [ ]:
# Save the results
results_csv_path = f'/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/esm/updated/{gene_name}_updated_deltaz_llr.csv'
results_df.to_csv(results_csv_path, index=False)

In [ ]:
# # Perform the merge operation based on the 'one_letter_mutation' column
# merged_df = pd.merge(results_df, deltaz_unique[['one_letter_mutation','gene','phenotype', 'type_label']], 
#                      on='one_letter_mutation', how='left')

# # Display the resulting merged DataFrame
# print(merged_df)


In [ ]:
results_df.columns

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load all deltaz_llr.csv files into a dictionary of DataFrames
genes = ['embB', 'ethA', 'gid', 'gyrA', 'inhA', 'katG', 'pncA', 'rpoB', 'rpsL']
data_frames = {}

for gene in genes:
    file_path = f'/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/esm/updated/{gene}_updated_deltaz_llr.csv'
    data_frames[gene] = pd.read_csv(file_path)

# Concatenate all DataFrames into one for aggregate analysis
combined_df = pd.concat(data_frames.values(), ignore_index=True)


In [ ]:
# Group by gene and phenotype (Resistant or Susceptible)
summary_stats = combined_df.groupby(['gene_name', 'phenotype'])['llr_score'].agg(['mean', 'median', 'min', 'max']).reset_index()

# Display the summary table
print(summary_stats)

# Save the summary table to a CSV file
summary_stats.to_csv('llr_summary_stats.csv', index=False)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Initialize a figure for the composite violin plot
plt.figure(figsize=(15, 10))

# Loop over each gene to create individual violin plots
for i, gene in enumerate(genes, 1):
    plt.subplot(3, 3, i)  # Adjust based on the number of genes (3x3 grid for 9 genes)
    # sns.violinplot(x='phenotype', y='llr_score', data=data_frames[gene], hue='phenotype', palette={'R': 'orange', 'S': 'blue'}, split=True)
    sns.violinplot(data=data_frames[gene], x="type_label", y="llr_score", hue="phenotype")
    plt.title(f'LLR Scores for {gene}')
    plt.xlabel('Phenotype')
    plt.ylabel('LLR Score')
    plt.legend([],[], frameon=False)  # Remove legend from individual subplots

# Adjust the layout and save the composite violin plot
plt.tight_layout()
plt.savefig('composite_llr_violin_plot.png')
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define the genes we want to plot
genes_to_plot = ['rpoB', 'katG', 'pncA']

# Filter out wildtype data, normalize LLR scores, and remove outliers
for gene in genes_to_plot:
    # Filter out wildtype entries
    data_frames[gene] = data_frames[gene][data_frames[gene]['type_label'] != 'wildtype']
    
    # Ensure the order of the phenotype categories is consistent across all genes
    data_frames[gene]['phenotype'] = pd.Categorical(data_frames[gene]['phenotype'], categories=['R', 'S'], ordered=True)
    
    # Identify any drastic outliers using the IQR (Interquartile Range) method
    Q1 = data_frames[gene]['llr_score'].quantile(0.25)
    Q3 = data_frames[gene]['llr_score'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Filter out the outliers
    data_frames[gene] = data_frames[gene][(data_frames[gene]['llr_score'] >= lower_bound) & (data_frames[gene]['llr_score'] <= upper_bound)]

# Combine the filtered data for normalization
combined_df = pd.concat([data_frames[gene] for gene in genes_to_plot], ignore_index=True)

# Normalize LLR scores across the selected genes
min_llr = combined_df['llr_score'].min()
max_llr = combined_df['llr_score'].max()

for gene in genes_to_plot:
    data_frames[gene]['llr_score'] = (data_frames[gene]['llr_score'] - min_llr) / (max_llr - min_llr)

# Define a consistent color palette where Resistant (R) is always blue and Susceptible (S) is always orange
palette = {'R': 'blue', 'S': 'orange'}

# Initialize a figure for the composite violin plot
plt.figure(figsize=(15, 5))  # Adjust size for 3 plots in a row

# Set fixed y-axis limits for uniformity across subplots
y_limits = (-0.2, 1.2)  # Adjust if needed

# Define subplot titles
# subplot_titles = ['rpoB: Resistant vs. Susceptible', 'katG: Resistant vs. Susceptible', 'embB: Resistant vs. Susceptible']
subplot_titles = [r'$rpoB$: Resistant vs. Susceptible', 
                  r'$katG$: Resistant vs. Susceptible', 
                  r'$pncA$: Resistant vs. Susceptible']

# Plot for each of the specified genes with consistent color mapping and explicit phenotype ordering
for i, gene in enumerate(genes_to_plot, 1):
    plt.subplot(1, 3, i)  # 1 row, 3 columns
    # sns.violinplot(data=data_frames[gene], x="type_label", y="llr_score", hue="phenotype", palette=palette, split=True)
    sns.violinplot(data=data_frames[gene], x="type_label", y="llr_score", hue="phenotype")
    plt.title(subplot_titles[i-1], fontsize=16)  # Use concise subplot titles
    plt.xlabel('Phenotype', fontsize=14)  # Increase x-axis label font size
    plt.ylabel('Normalized LLR Score', fontsize=14)  # Increase y-axis label font size
    plt.ylim(y_limits)  # Set uniform y-axis limits
    plt.xticks(fontsize=18)  # Increase x-axis tick font size
    plt.yticks(fontsize=18)  # Increase y-axis tick font size

    # Add the same consistent legend for each plot
    plt.legend(title='Phenotype', labels=['Resistant', 'Susceptible'], loc='upper right', fontsize=12)

# Add the figure caption
plt.figtext(0.5, -0.05, 'LLR score distribution for distinguishing resistant (R) and susceptible (S) phenotypes based on embeddings derived from the ESM-2 model.', ha='center', fontsize=14)

# Adjust the layout and save the composite violin plot
plt.tight_layout()
plt.savefig('composite_llr_violin_plot_rpoB_katG_embB_consistent_colors_fixed_order_caption.png', bbox_inches='tight')
plt.show()


In [ ]:
# Initialize a figure for the composite violin plot
plt.figure(figsize=(15, 10))

# Loop over each gene to create individual violin plots
for i, gene in enumerate(genes, 1):
    plt.subplot(3, 3, i)  # Adjust based on the number of genes (3x3 grid for 9 genes)
    sns.violinplot(x='phenotype', y='llr_score', data=data_frames[gene], hue='phenotype', split=True, palette={'R': 'orange', 'S': 'blue'})
    plt.title(f'LLR Scores for {gene}')
    plt.xlabel('Phenotype')
    plt.ylabel('LLR Score')
    plt.legend(loc='upper right')

# Adjust the layout and save the composite violin plot
plt.tight_layout()
plt.savefig('composite_llr_violin_plot_colored.png')
plt.show()


In [ ]:
import pandas as pd

In [ ]:
# gene_name='rpoB'
file_path = f'/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/esm/updated/{gene_name}_updated_deltaz_llr.csv'
df = pd.read_csv(file_path)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def calculate_auc(df, gene_name):
    # Filter data for the specific gene
    gene_df = df[df['gene_name'] == gene_name].dropna(subset=['llr_score', 'phenotype'])

    # Encode the phenotype (Assume 'R' for resistant, 'S' for susceptible)
    gene_df['target'] = gene_df['phenotype'].apply(lambda x: 1 if x == 'R' else 0)

    # Define the features and target
    X = gene_df[['llr_score']]
    y = gene_df['target']

    # Split data into train and test sets
    # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    print(f"Class distribution in y_test: {np.bincount(y_test)}")

    # Initialize and fit logistic regression model
    log_reg = LogisticRegression()
    log_reg.fit(X_train, y_train)

    # Predict probabilities on the test set
    y_pred_prob = log_reg.predict_proba(X_test)[:, 1]  # Probability of the positive class

    # Calculate AUC
    auc_value = roc_auc_score(y_test, y_pred_prob)
    print(f"AUC for gene {gene_name}: {auc_value}")

    # Plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_value:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic for {gene_name}')
    plt.legend(loc="lower right")
    plt.show()

    return auc_value



In [ ]:
# Example usage
auc_value = calculate_auc(df, gene_name)


In [ ]:
data_table_df= pd.DataFrame(data_table)
data_table_df

In [ ]:
data_table_df.to_csv(f'/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/esm/data_table.csv', index=False)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, auc

# List of genes
genes = ['embB', 'ethA', 'gid', 'gyrA', 'inhA', 'katG', 'pncA', 'rpoB','rpsL']

# Dictionary to store AUC results
auc_results = {}

# Function to calculate AUC for each gene
def calculate_auc_for_gene(df, gene_name):
    # Ensure necessary columns are present and no missing data
    df = df.dropna(subset=['llr_score', 'phenotype'])
    
    # Encode phenotype: 'R' as 1 and 'S' as 0
    df['target'] = df['phenotype'].apply(lambda x: 1 if x == 'R' else 0)
    
    # Define the features (LLR scores) and target (phenotype)
    X = df[['llr_score']]
    y = df['target']

    # Split data into train and test sets
    # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    # Initialize and fit logistic regression model
    log_reg = LogisticRegression()
    log_reg.fit(X_train, y_train)
    
    # Predict probabilities on the test set
    y_pred_prob = log_reg.predict_proba(X_test)[:, 1]  # Probability of the positive class
    
    # Calculate AUC
    auc_value = roc_auc_score(y_test, y_pred_prob)
    print(f"AUC for gene {gene_name}: {auc_value}")
    # Plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_value:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic for {gene_name}')
    plt.legend(loc="lower right")
    plt.show()

    return auc_value



In [ ]:
# Iterate over each gene, calculate AUC, and store the result
for gene in genes:
    df = data_frames[gene]
    auc = calculate_auc_for_gene(df, gene)
    auc_results[gene] = auc

# Convert results to a DataFrame
auc_df = pd.DataFrame(list(auc_results.items()), columns=['Gene', 'AUC'])

auc_df

In [ ]:
# Save the results to a CSV file
auc_df.to_csv('/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/esm/llr_auc_results.csv', index=False)

# Display the AUC results
print(auc_df)


# supervised variant prediction

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import scipy.stats
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

In [ ]:
import torch
import esm
import pandas as pd
import gc
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set your local ESM model path
MODEL_PATH = "/datasets/bio/esm/models/esm2_t6_8M_UR50D.pt"

def identify_mutations(seq, ref_seq):
    """
    Identifies mutations by comparing the sequence to the reference wild-type protein.
    """
    mutations = []
    min_length = min(len(seq), len(ref_seq))  # Handle different lengths safely

    for i in range(min_length):
        if seq[i] != ref_seq[i]:  # Mutation detected
            mutations.append(f"p.{ref_seq[i]}{i+1}{seq[i]}")  # Format: p.A23T

    return mutations

def generate_embeddings(df, ref_protein, model_path=MODEL_PATH, include=["mean"], truncate_len=1022):
    """
    Extracts embeddings and dynamically identifies mutations without re-downloading the model.

    Args:
        df (pd.DataFrame): DataFrame containing sequences.
        ref_protein (str): Wild-type reference protein sequence.
        model_path (str): Path to locally stored ESM model.
        include (list): List of representations to extract. Options: ["mean", "per_tok", "bos"].
        truncate_len (int): Maximum sequence length before truncation.

    Returns:
        pd.DataFrame: DataFrame containing embeddings and mutations.
    """
    # Load Model from local path
    model, alphabet = esm.pretrained.load_model_and_alphabet_local(model_path)
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Initialize batch converter
    batch_converter = alphabet.get_batch_converter()

    # Prepare storage lists
    embeddings_list = []
    per_token_representations = []
    bos_representations = []
    labels = []
    mutations_all = []

    # Process sequences in batches
    data = list(zip(df["Filename"], df["Protein_Sequence"]))  # Assuming `Filename` is the identifier
    dataloader = DataLoader(data, batch_size=1, shuffle=False)

    with torch.no_grad():
        for labels_batch, sequences_batch in tqdm(dataloader, desc="Processing Batches"):
            batch_labels, batch_strs, batch_tokens = batch_converter(list(zip(labels_batch, sequences_batch)))
            batch_tokens = batch_tokens.to(device)

            # Forward pass
            model_results = model(batch_tokens, repr_layers=[model.num_layers], return_contacts=False)
            token_representations = model_results["representations"][model.num_layers].to("cpu")

            for i, seq in enumerate(batch_strs):
                seq_len = min(truncate_len, len(seq))

                # Mean-pooled embedding
                if "mean" in include:
                    mean_repr = token_representations[i, 1:seq_len+1].mean(0).clone()
                    embeddings_list.append(mean_repr)

                # Per-token representations
                if "per_tok" in include:
                    per_tok_repr = token_representations[i, 1:seq_len+1].clone()
                    per_token_representations.append(per_tok_repr)

                # BOS-token representation
                if "bos" in include:
                    bos_repr = token_representations[i, 0].clone()
                    bos_representations.append(bos_repr)

                # Track labels and mutations
                labels.append(batch_labels[i])

                # **Identify mutations dynamically**
                mutations = identify_mutations(seq, ref_protein)
                mutations_all.append(mutations)
            torch.cuda.empty_cache()
            gc.collect()

    # Convert embeddings to DataFrame
    df_embeddings = pd.DataFrame({
        "identifier": labels,
        "mean_embedding": embeddings_list if "mean" in include else None,
        "per_token_embedding": per_token_representations if "per_tok" in include else None,
        "bos_embedding": bos_representations if "bos" in include else None,
        "mutation": mutations_all  # **Now dynamically computed!**
    })

    # Clean up memory
    gc.collect()
    torch.cuda.empty_cache()

    return df_embeddings


## data generation

In [ ]:
gene_name='gid'

In [ ]:
ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')
ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene_name, 'protein_sequence'].values

In [ ]:
gene_sequences=pd.read_csv(f"./data/sequence_data_csv/{gene_name}_sequence_data.csv")

In [ ]:
gene_sequences_filtered = gene_sequences[gene_sequences["Frameshift_Mutation"] == 0].copy()

In [ ]:
# Ensure required columns exist
if "Protein_Sequence" not in gene_sequences_filtered.columns:
    raise ValueError("CSV file must contain a 'Protein_Sequence' column.")

In [ ]:
valid_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")  # Standard 20 amino acids

# Check each sequence
for idx, seq in enumerate(gene_sequences_filtered["Protein_Sequence"]):
    invalid_chars = [c for c in seq if c not in valid_amino_acids]
    if invalid_chars:
        print(f"WARNING: Sequence {idx} contains invalid characters: {invalid_chars}")


In [ ]:
chunk_size = 1000  # Adjust based on memory availability
for i in range(0, len(gene_sequences_filtered), chunk_size):
    df_chunk = gene_sequences_filtered.iloc[i:i+chunk_size]
    df_embeddings = generate_embeddings(df_chunk, ref_protein, include=["mean"])
    
    # Save or append results immediately to avoid memory buildup
    df_embeddings.to_csv(f"{gene_name}_embeddings_chunk_{i}.csv", index=False)
    # Clean up memory after each chunk!
    del df_embeddings
    torch.cuda.empty_cache()
    gc.collect()



In [ ]:
#combine all embedding files

import pandas as pd
import glob

# Find all embedding CSV files (update the pattern if needed)
embedding_files = glob.glob(f"{gene_name}_embeddings_chunk_*.csv")  
# Load and combine all CSVs
dfs = [pd.read_csv(f) for f in embedding_files]
combined_df = pd.concat(dfs, ignore_index=True)

# Save the combined embeddings file
combined_df.to_csv(f"{gene_name}_all_embeddings.csv", index=False)

print(f"Combined {len(embedding_files)} embedding files into 'all_embeddings.csv'.")
print("Shape of combined embeddings:", combined_df.shape)


In [ ]:
# merge with phenotype
import pandas as pd
import torch

# Load embeddings CSV
embeddings_df = pd.read_csv(f"{gene_name}_all_embeddings.csv")  # Update with actual path


# Load sequences CSV (which contains phenotypes)
sequences_df=pd.read_csv(f"./data/sequence_data_csv/{gene_name}_sequence_data.csv")
sequences_df = sequences_df[sequences_df["Frameshift_Mutation"] == 0].copy()

In [ ]:
# Merge on Filename (sequences) and identifier (embeddings)
merged_df = embeddings_df.merge(sequences_df[['Filename', 'Phenotype']], 
                                left_on="identifier", right_on="Filename", 
                                how="inner")

# Drop redundant Filename column after merging
merged_df.drop(columns=["Filename"], inplace=True)

In [ ]:
# Save the merged dataset
merged_df.to_csv(f"{gene_name}_merged_embeddings_with_phenotype.csv", index=False)

print(f"Merged embeddings with phenotype. Final shape: {merged_df.shape}")

## train model

In [ ]:
import pandas as pd
import numpy as np
import ast
import torch
import scipy.stats
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

# Directory to save results
RESULTS_DIR = "gene_results"
os.makedirs(RESULTS_DIR, exist_ok=True)


#  Step 1: Convert String Embeddings to Numerical Arrays
def parse_tensor(tensor_str):
    """Convert string tensor representation into a NumPy array."""
    try:
        return np.array(ast.literal_eval(tensor_str.replace("tensor(", "").replace(")", "")))
    except Exception as e:
        print(f"Error parsing tensor: {tensor_str} -> {e}")
        return np.zeros(320)  # Adjust size if needed


#  Step 2: Load and Merge Data
def load_embeddings(df):
    """
    Load embeddings from a DataFrame and convert to numerical arrays.
    """
    df["mean_embedding"] = df["mean_embedding"].apply(parse_tensor)
    return df



#  Step 3: Extract Features (X) and Target (y)
def prepare_data(df):
    """
    Prepare feature matrix (X) and target labels (y) for training.
    Filters for binary classification: only 'R' and 'S' phenotypes.
    """
    # Filter to only 'R' and 'S'
    df_binary = df[df["Phenotype"].isin(["R", "S"])].copy()

    # Encode labels (R -> 1, S -> 0)
    y = df_binary["Phenotype"].astype("category").cat.codes
    X = np.vstack(df_binary["mean_embedding"].values)

    print(f"Binary classification only. Data prepared. Shape X: {X.shape}, y: {y.shape}, Classes: {np.unique(y)}")
    return X, y


#  Step 4: Train-Test Split
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)


#  Step 5: PCA Visualization and Save
def visualize_pca(X_train, y_train, gene_name, num_components=60):
    """
    Perform PCA and save the visualization.
    """
    pca = PCA(n_components=num_components)
    X_train_pca = pca.fit_transform(X_train)

    plt.figure(figsize=(7, 6))
    sc = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, marker='.')
    plt.xlabel("PCA First Principal Component")
    plt.ylabel("PCA Second Principal Component")
    plt.colorbar(sc, label="Variant Effect")
    plt.title(f"PCA Visualization for {gene_name}")

    # Save PCA Figure
    pca_path = os.path.join(RESULTS_DIR, f"{gene_name}_pca.png")
    plt.savefig(pca_path)
    plt.close()
    
    return pca


#  Step 6: Hyperparameter Tuning using GridSearchCV
def run_grid_search(X_train, y_train, num_pca_components=60):
    """
    Perform hyperparameter tuning using GridSearchCV on KNN, SVM, and Random Forest.
    """
    knn_grid = [
        {
            'model': [KNeighborsRegressor()],
            'model__n_neighbors': [5, 10],
            'model__weights': ['uniform', 'distance'],
            'model__algorithm': ['ball_tree', 'kd_tree', 'brute'],
            'model__leaf_size': [15, 30],
            'model__p': [1, 2],
        }
    ]

    svm_grid = [
        {
            'model': [SVR()],
            'model__C': [0.1, 1.0, 10.0],
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__degree': [3],
            'model__gamma': ['scale'],
        }
    ]

    rfr_grid = [
        {
            'model': [RandomForestRegressor()],
            'model__n_estimators': [20],
            'model__criterion': ['squared_error', 'absolute_error'],
            'model__max_features': ['sqrt', 'log2'],
            'model__min_samples_split': [5, 10],
            'model__min_samples_leaf': [1, 4]
        }
    ]

    cls_list = [KNeighborsRegressor, SVR, RandomForestRegressor]
    param_grid_list = [knn_grid, svm_grid, rfr_grid]

    pipe = Pipeline(
        steps=[('pca', PCA(num_pca_components)), ('model', 'passthrough')]
    )

    result_list = []
    grid_list = []

    for cls_name, param_grid in zip(cls_list, param_grid_list):
        print(f"Running Grid Search for {cls_name.__name__}...")

        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring='r2',
            verbose=1,
            n_jobs=-1
        )
        grid.fit(X_train, y_train)
        result_list.append(pd.DataFrame.from_dict(grid.cv_results_))
        grid_list.append(grid)

    return result_list, grid_list


# Step 7: Evaluate Best Models and Save Results
def evaluate_models_spearman(grid_list, X_test, y_test, gene_name):
    """
    Evaluate the best models using Spearman correlation and save results.
    """
    results = []
    
    for grid in grid_list:
        best_model = grid.best_estimator_.get_params()["steps"][1][1]
        preds = grid.predict(X_test)
        spearman_corr, _ = scipy.stats.spearmanr(y_test, preds)

        # Save results
        results.append({
            "Gene": gene_name,
            "Best Model": str(best_model),
            "Spearman Correlation": spearman_corr
        })

        print(f" Best Model for {gene_name}: {best_model}")
        print(f"Spearman Correlation: {spearman_corr:.4f}")
        print("\n", "-" * 80, "\n")

    return results


from sklearn.metrics import roc_auc_score

#  Step 7: Evaluate Best Models using AUC and Save Results
def evaluate_models(grid_list, X_test, y_test, gene_name):
    """
    Evaluate the best models using AUC and save results.
    """
    results = []

    for grid in grid_list:
        best_model = grid.best_estimator_.get_params()["steps"][1][1]
        preds = grid.predict(X_test)

        try:
            auc = roc_auc_score(y_test, preds)
        except ValueError as e:
            print(f"[Warning] AUC could not be computed for {gene_name}: {e}")
            auc = None

        results.append({
            "Gene": gene_name,
            "Best Model": str(best_model),
            "AUC": auc
        })

        print(f"Best Model for {gene_name}: {best_model}")
        if auc is not None:
            print(f"AUC: {auc:.4f}")
        else:
            print("AUC: N/A (only one class present in test set)")
        print("\n", "-" * 80, "\n")

    return results



In [ ]:
# Step 8: Run for Each Gene and Save All Results
def main(gene_df):
    """
    Run the pipeline for each gene and store results in CSV.
    """
    all_results = []


    print(f"Processing gene: {gene_name} ({len(gene_df)} samples)")

    df_gene = load_embeddings(gene_df)
    X, y = prepare_data(df_gene)
    X_train, X_test, y_train, y_test = split_data(X, y)
    print(y_test)

    # PCA and save visualization
    visualize_pca(X_train, y_train, gene_name)

    # Hyperparameter tuning
    _, grid_list = run_grid_search(X_train, y_train)

    # Evaluate and collect results
    results = evaluate_models(grid_list, X_test, y_test, gene_name)
    all_results.extend(results)

    # Save all results to CSV
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(os.path.join(RESULTS_DIR, f"{gene_name}_model_performance.csv"), index=False)

    print(f"\n All results saved in '{RESULTS_DIR}/model_performance.csv'")



In [ ]:

# Run the pipeline with the gene dataframe
gene_name='rpoB'
merged_df=pd.read_csv(f"./data/embeddings/{gene_name}_merged_embeddings_with_phenotype.csv")


In [ ]:
len(merged_df['mean_embedding'][0])

In [ ]:
len(merged_df['mean_embedding'][1])

In [ ]:
main(merged_df)

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Define the directory containing the CSV files
directory_path = "./esm/gene_results" 

# Load all *_model_performance.csv files
all_files = glob.glob(os.path.join(directory_path, "*_model_performance.csv"))

# Check if any files were found
if not all_files:
    raise FileNotFoundError("No model_performance.csv files found in the specified directory.")

# Read each file and store valid dataframes
df_list = []
for file in all_files:
    try:
        df_temp = pd.read_csv(file)
        df_temp["Gene"] = os.path.basename(file).split("_")[0]  # Extract gene name from filename
        df_list.append(df_temp)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Ensure we have valid data
if not df_list:
    raise ValueError("No valid, non-empty CSV files were found.")

# Concatenate all dataframes into a single dataframe
df = pd.concat(df_list, ignore_index=True)

# Ensure column names are correctly formatted
df.columns = df.columns.str.strip()

# Plotting
plt.figure(figsize=(12, 6))

# Using different colors for each gene
genes = df["Gene"].unique()
colors = plt.get_cmap("tab10")  # Define colormap

for i, gene in enumerate(genes):
    subset = df[df["Gene"] == gene]
    plt.barh(subset["Best Model"], subset["AUC"], label=gene, color=colors(i / len(genes)), alpha=0.7)

plt.xlabel("Spearman Correlation")
plt.ylabel("Best Model")
plt.title("Model Performance for Different Genes")
plt.xlim(0, 1)
plt.legend(title="Gene", loc="lower right")
plt.gca().invert_yaxis()

# Show the plot
plt.show()


In [ ]:
# Extract unique genes
unique_genes = df_filtered["Gene"].unique()


# Assign numerical values to genes for x-axis
gene_to_x = {gene: i + 1 for i, gene in enumerate(unique_genes)}
df_filtered["Gene X"] = df_filtered["Gene"].map(gene_to_x)

# Plotting
plt.figure(figsize=(12, 6))

# Colors for different models
colors = {"KNN": "blue", "SVR": "orange", "Random Forest": "green"}

for model in df_filtered["Model Type"].unique():
    subset = df_filtered[df_filtered["Model Type"] == model]
    plt.errorbar(
        subset["Gene X"], subset["AUC"],
        yerr=subset["Error"], fmt="o", label=model, color=colors[model], capsize=4
    )

plt.axhline(y=0.5, color="red", linestyle="dashed")  # Reference line
plt.xticks(list(gene_to_x.values()), list(gene_to_x.keys()), rotation=45)  # Set gene labels on x-axis
plt.xlabel("Genes")
plt.ylabel("AUC")
plt.title("Model Performance for Each Gene: ESM Supervised Variant Prediction Task")
plt.legend(title="Model Type", loc="upper left")
plt.ylim(0.0, 1)

# Show the plot
plt.show()


In [ ]:
print(df.columns.tolist())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Clean column names
df.columns = df.columns.str.strip()

# Step 2: Map full model descriptions to simplified labels
def simplify_model_name(full_model_name):
    if "KNeighborsRegressor" in full_model_name:
        return "KNN"
    elif "SVR" in full_model_name:
        return "SVR"
    elif "RandomForestRegressor" in full_model_name:
        return "Random Forest"
    else:
        return "Other"

df["Model Type"] = df["Best Model"].apply(simplify_model_name)

# Step 3: Filter only the desired model types
df_filtered = df[df["Model Type"].isin(["KNN", "SVR", "Random Forest"])].copy()

# Step 4: Group by Gene and Model Type to get mean AUC and std as error
df_grouped = df_filtered.groupby(["Gene", "Model Type"]).agg(
    AUC=("AUC", "mean"),
    Error=("AUC", "std")
).reset_index()

# Step 5: Assign x-axis positions to each gene
unique_genes = df_grouped["Gene"].unique()
gene_to_x = {gene: i + 1 for i, gene in enumerate(unique_genes)}
df_grouped["Gene X"] = df_grouped["Gene"].map(gene_to_x)

# Step 6: Plotting
plt.figure(figsize=(12, 6))

colors = {"KNN": "blue", "SVR": "orange", "Random Forest": "green"}

for model in ["KNN", "SVR", "Random Forest"]:
    subset = df_grouped[df_grouped["Model Type"] == model]
    plt.errorbar(
        subset["Gene X"], subset["AUC"],
        yerr=subset["Error"], fmt="o", label=model,
        color=colors[model], capsize=4
    )

plt.axhline(y=0.5, color="red", linestyle="dashed")  # Reference line
plt.xticks(list(gene_to_x.values()), list(gene_to_x.keys()), rotation=45)
plt.xlabel("Genes")
plt.ylabel("AUC")
plt.title("Model Performance for Each Gene: ESM Supervised Variant Prediction Task")
plt.legend(title="Model Type", loc="lower left")
plt.ylim(0.0, 1)

plt.tight_layout()
plt.show()


In [ ]:
# File paths (adjust as needed)
combined_files = {
    "katG+inhA": "./esm/gene_results/katg_inhA_model_performance.csv",
    "ethA+inhA": "./esm/gene_results/ethA_inhA_model_performance.csv"
}

# Load and tag the combined gene CSVs
combined_dfs = []
for gene_name, filepath in combined_files.items():
    try:
        df_comb = pd.read_csv(filepath)
        df_comb["Gene"] = gene_name
        combined_dfs.append(df_comb)
    except Exception as e:
        print(f"Error loading {gene_name}: {e}")


In [ ]:
# Combine all dataframes
df_all = pd.concat([df] + combined_dfs, ignore_index=True)

# Clean and standardize column names
df_all.columns = df_all.columns.str.strip().str.lower().str.replace(" ", "_")


In [ ]:
# Ensure consistent model name extraction
def simplify_model_name(full_model_name):
    if "KNeighborsRegressor" in full_model_name:
        return "KNN"
    elif "SVR" in full_model_name:
        return "SVR"
    elif "RandomForestRegressor" in full_model_name:
        return "Random Forest"
    else:
        return "Other"

df_all["model_type"] = df_all["best_model"].apply(simplify_model_name)


In [ ]:
# Filter valid models
df_filtered = df_all[df_all["model_type"].isin(["KNN", "SVR", "Random Forest"])].copy()

# Aggregate
df_grouped = df_filtered.groupby(["gene", "model_type"]).agg(
    auc=("auc", "mean"),
    error=("auc", "std")
).reset_index()

# Map gene names to x-axis
unique_genes = df_grouped["gene"].unique()
gene_to_x = {gene: i + 1 for i, gene in enumerate(unique_genes)}
df_grouped["gene_x"] = df_grouped["gene"].map(gene_to_x)

# Plot
plt.figure(figsize=(12, 6))
colors = {"KNN": "blue", "SVR": "orange", "Random Forest": "green"}

for model in ["KNN", "SVR", "Random Forest"]:
    subset = df_grouped[df_grouped["model_type"] == model]
    plt.errorbar(
        subset["gene_x"], subset["auc"],
        yerr=subset["error"], fmt="o", label=model,
        color=colors[model], capsize=4
    )

plt.axhline(y=0.5, color="red", linestyle="dashed")
plt.xticks(list(gene_to_x.values()), list(gene_to_x.keys()), rotation=45)
plt.xlabel("Genes")
plt.ylabel("AUC")
plt.title("Model Performance for Each Gene and Combined Genes")
plt.legend(title="Model Type", loc="lower left")
plt.ylim(0.0, 1)
plt.tight_layout()
plt.show()
